# Urdu Orpheus — stage-2 conversational adaptation (ASLP + UrduTTS), one notebook for every epoch
Trains on the curated **44 h** mixture in the private dataset `hassan-31x/urdu-orpheus-stage2-data` (30% ASLP conversational, 70% UrduTTS; fixed train/validation split, no clip repeated). The Trainer reshuffles the training set each epoch.

**Setup:** enable Internet + GPU and attach the `HF_TOKEN` Kaggle secret (read access to the dataset, write access to the checkpoint repo). Nothing else is needed.

**Workflow**
1. `TEST=True`, `EPOCH=2`: a few-clip end-to-end smoke run (training, checkpoint, Hub backup, samples). It writes a durable receipt for this commit.
2. `TEST=False`, `EPOCH=2`: the full second epoch, starting from the finished UrduTTS epoch-1 adapter (`checkpoint-5029`).
3. **Out of GPU hours?** Run the same notebook (same `REPO_REF`, same `EPOCH`) from another Kaggle account whose `HF_TOKEN` secret can reach the same Hub repos. It resumes from the latest verified checkpoint, optimizer and data position included.
4. **Next epoch:** when the epoch finishes, change only `EPOCH` (3, 4, ...). Each epoch starts from the previous epoch's finished adapter with a fresh optimizer, a new shuffle seed and the learning rate in `stage2_plan.py`; it reuses the previous epoch's SNAC token cache so nothing is re-encoded.

Keep `REPO_REF` pinned to one commit for all epochs so token caches and resumes stay valid. Progress, checkpoints and logs live in `HF_REPO_ID` under `runs/<run id>/`.


Cell 1 — clone a fixed version

In [ ]:
import os, sys, subprocess, re
from pathlib import Path

REPO_URL = "https://github.com/hassan-31x/orpheus-ft-aslp-urdutts.git"
REPO_REF = "PINNED_BY_PUSH_SCRIPT"  # Pinned commit; keep identical for every epoch and every account.
PROJECT_FOLDER = "."  # This standalone GitHub repository contains the scripts at its root.
HF_REPO_ID = "hassan-31x/orpheus-urdu-stage2-checkpoints"  # Private Hub MODEL repo holding every stage-2 epoch run.
HF_SECRET_NAME = "HF_TOKEN"  # Exact Kaggle secret LABEL used in your previous notebook; change if different.
EPOCH = 2  # 2 = from UrduTTS epoch 1; 3, 4, ... = from the finished previous stage-2 epoch.
LEARNING_RATE = None  # None uses the per-epoch value in stage2_plan.py. Keep fixed within one epoch.
TEST = True  # True: few-clip smoke run of this epoch. Set False after "TEST PASSED" for the full epoch.
TEST_TRAIN_CLIPS = 24
TEST_VALIDATION_CLIPS = 8
AUDIT_ONLY = False
QUALITY_DEVICE = "cuda"
SESSION_HOURS = 11.75  # Kaggle kills sessions at 12 h; training saves + uploads a checkpoint before that.
CLONE = Path("/kaggle/working/orpheus-mixed-code")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", HF_REPO_ID):
    raise ValueError("HF_REPO_ID must be username/repository")
def load_hf_token(secret_name="HF_TOKEN", environ=None, secret_client_factory=None):
    # Keep a valid explicit environment token; do not overwrite it with a failed Kaggle lookup.
    environ = os.environ if environ is None else environ
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = environ.get(name, "").strip()
        if value:
            environ["HF_TOKEN"] = value
            return "environment:" + name
    for _path in sorted(Path("/kaggle/input").glob("**/hf_token.txt")):
        value = _path.read_text().strip()
        if value:
            environ["HF_TOKEN"] = value
            return "attached private Kaggle dataset"
    if not isinstance(secret_name, str) or not secret_name.strip():
        raise RuntimeError("HF_SECRET_NAME must be the exact label in Kaggle Add-ons > Secrets")
    try:
        if secret_client_factory is None:
            from kaggle_secrets import UserSecretsClient
            secret_client_factory = UserSecretsClient
        import time as _time
        for _attempt in range(3):  # the secrets service can refuse connections in API-started sessions
            try:
                value = secret_client_factory().get_secret(secret_name)
                break
            except ConnectionError:
                if _attempt == 2:
                    raise
                _time.sleep(10)
    except Exception as exc:
        # Never display exception text: upstream responses can contain credentials.
        error_type = type(exc).__name__
        raise RuntimeError(
            "Kaggle could not read the Hugging Face secret (" + error_type + "). "
            "Open this notebook in Edit mode > Add-ons > Secrets. Add/select the secret "
            "whose label exactly matches HF_SECRET_NAME and ENABLE its access for THIS notebook. "
            ""
            "Use a Hugging Face token with write access to " + HF_REPO_ID + ". "
            "Then Save Version > Save & Run All again. Do not paste the token into notebook code."
        ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError("The selected Kaggle secret is empty; update its value in Add-ons > Secrets")
    environ["HF_TOKEN"] = value.strip()
    return "Kaggle Secrets"

credential_source = load_hf_token(HF_SECRET_NAME)
print("Hugging Face credential loaded from", credential_source, "(value hidden)")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
if not AUDIT_ONLY or QUALITY_DEVICE == "cuda":
    try:
        gpu_list = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        gpu_list = ""
    if "GPU" not in gpu_list:
        raise RuntimeError("Enable an NVIDIA GPU in Kaggle Settings; check remaining GPU quota")
    print(gpu_list.strip())
if not CLONE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=CLONE, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=CLONE, check=True)
REPO = CLONE / PROJECT_FOLDER if (CLONE / PROJECT_FOLDER).is_dir() else CLONE
for name in ("setup_kaggle.py", "finetune_mixed.py", "stage2_data.py", "stage2_plan.py", "checkpoint_init.py", "urdutts_data.py", "audio_storage.py", "workflow_gate.py", "configs/mixed.json", "configs/stage2.json"):
    if not (REPO / name).is_file():
        raise RuntimeError(f"Missing {name}: push the mixed-data folder and check REPO_REF/PROJECT_FOLDER")
PROJECT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE, text=True).strip()
print("Project commit (pin REPO_REF to this for resumes):", PROJECT_COMMIT)
early_check = subprocess.run([sys.executable, str(REPO / "check_hub.py"), "--repo", HF_REPO_ID], cwd=REPO)
if early_check.returncode == 2:
    raise RuntimeError("Hugging Face storage check failed; see redacted diagnostics above")
if early_check.returncode:
    print("Early Hub check unavailable in host Python; it will run in the installed training venv")

Cell 2 — install and preflight

In [ ]:
# Dedicated venv inherits Kaggle CUDA wheels; installs project dependencies every fresh session.
ENV = Path("/kaggle/working/orpheus-mixed-env")
TRAIN_PYTHON = str(ENV / "bin" / "python")
setup_command = [sys.executable, "-u", str(REPO / "setup_kaggle.py"), "--venv", str(ENV)]
if not AUDIT_ONLY:
    setup_command.append("--gpu")
subprocess.run(setup_command, cwd=REPO, check=True)
subprocess.run([TRAIN_PYTHON, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=REPO, check=True)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)

Cell 3 — private credentials and config

In [ ]:
import json
sys.path.insert(0, str(REPO))
import stage2_plan
base = stage2_plan.load_base(REPO)
cfg = stage2_plan.epoch_config(EPOCH, base, LEARNING_RATE)
if TEST:
    if TEST_TRAIN_CLIPS < 2 or TEST_VALIDATION_CLIPS < 2:
        raise ValueError("Smoke subsets need at least two clips")
    cfg["run_id"] += f"-smoke-{TEST_TRAIN_CLIPS}-{TEST_VALIDATION_CLIPS}-{PROJECT_COMMIT[:7]}"
    cfg.update(test_train_clips=TEST_TRAIN_CLIPS, test_validation_clips=TEST_VALIDATION_CLIPS,
               save_steps=2, eval_steps=2, logging_steps=1, sample_count=1)
cfg.update(checkpoint_backend="huggingface", hf_repo_id=HF_REPO_ID, session_hours=SESSION_HOURS)
CONFIG = Path("/kaggle/working/stage2-run-config.json")
CONFIG.write_text(json.dumps(cfg, indent=2))
RUN = Path(cfg["work_dir"]) / "runs" / cfg["run_id"]
print("Run:", cfg["run_id"], "| epoch:", EPOCH, "| TEST:", TEST, "| lr:", cfg["learning_rate"], "| seed:", cfg["seed"])
print("Initialisation:", cfg["init_from_run"] or cfg["init_source"])
print("Data:", cfg["dataset_id"], "revision:", cfg["dataset_revision"][:10])
subprocess.run([TRAIN_PYTHON, str(REPO / "finetune_mixed.py"),
                "--config", str(CONFIG), "--mode", "storage-check"], cwd=REPO, check=True)
# Full training needs a durable smoke receipt for this exact commit and epoch config (run TEST=True first).
from workflow_gate import source_and_smoke_gate
source_and_smoke_gate(cfg, TEST or AUDIT_ONLY, PROJECT_COMMIT)
CONFIG.write_text(json.dumps(cfg, indent=2))


In [ ]:
# Resuming on a fresh Kaggle machine: the venv resolves unpinned dependencies again, but the saved
# checkpoint's identity records exact versions. Align the venv to the versions the run started with.
import re as _re, json as _json
from huggingface_hub import HfApi, hf_hub_download
_prefix = f"runs/{cfg['run_id']}/snapshots/"
_snaps = {}
for _f in HfApi().list_repo_files(HF_REPO_ID):
    _m = _re.fullmatch(_re.escape(_prefix) + r"(checkpoint-(\d+)-[0-9a-f]+)/environment\.json", _f)
    if _m:
        _snaps[int(_m.group(2))] = _f
if _snaps and not TEST:
    _saved = _json.load(open(hf_hub_download(HF_REPO_ID, _snaps[max(_snaps)])))
    _probe = "import importlib.metadata as m,json,sys;print(json.dumps({n:m.version(n) for n in sys.argv[1:]}))"
    _now = _json.loads(subprocess.check_output([TRAIN_PYTHON, "-c", _probe, *_saved["packages"]], text=True))
    _diff = {n: (v, _now.get(n)) for n, v in _saved["packages"].items() if _now.get(n) != v}
    print("Package drift vs saved run (saved, now):", _diff or "none")
    if _diff:
        subprocess.run([TRAIN_PYTHON, "-m", "pip", "install", "--no-deps", *[f"{n}=={v}" for n, (v, _) in _diff.items()]], check=True)
    import torch as _t
    print("torch saved:", _saved["torch"])


Cell 4A — recommended unattended saved run

In [ ]:
# Blocks until the audit/training process completes. Failures print diagnostics.
command = [TRAIN_PYTHON, "-u", str(REPO / "finetune_mixed.py"), "--config", str(CONFIG)]
if AUDIT_ONLY:
    command += ["--mode", "audit"]
import time

CONSOLE = Path("/kaggle/working/mixed_train_console.log")

def show_tail(lines):
    try:
        with CONSOLE.open("rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 200_000))
            print("\n".join(f.read().decode("utf-8", "replace").splitlines()[-lines:]), flush=True)
    except OSError:
        pass

# Training output goes to a FILE, never the notebook stream: a stalled Kaggle log
# pipeline can block a process that writes to it. Progress tail printed every 5 min;
# full log: /kaggle/working/mixed_train_console.log (and monitoring/ on Hugging Face).
print("Training output ->", CONSOLE, flush=True)
try:
    with CONSOLE.open("ab") as console:
        process = subprocess.Popen(command, cwd=REPO, stdout=console, stderr=subprocess.STDOUT,
                                   stdin=subprocess.DEVNULL)
        last_tail = time.time()
        while process.poll() is None:
            time.sleep(15)
            if time.time() - last_tail >= 300:
                print(f"--- {time.strftime('%H:%M:%S')} last lines ---", flush=True)
                show_tail(6)
                last_tail = time.time()
    show_tail(60)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)
except subprocess.CalledProcessError:
    import shutil
    print("Free disk GiB:", round(shutil.disk_usage(RUN).free / 2**30, 2))
    path = RUN / "disk_budget.json"
    if path.exists():
        print("Disk budget:", path.read_text())
    reclaim = RUN / "audio_storage.json"
    if reclaim.exists():
        print("Audio storage recovery:", reclaim.read_text())
    print("Training process exited. The traceback above is the failure; audit exclusions are separate.")
    if TEST:
        print("TEST FAILED. Keep TEST=True and fix the error above before a full run.")
    raise
if TEST and not AUDIT_ONLY:
    final_status = json.loads((RUN / "status.json").read_text()) if (RUN / "status.json").exists() else {}
    backup_info = json.loads((RUN / "backup_status.json").read_text()) if (RUN / "backup_status.json").exists() else {}
    wavs = sorted((RUN / "samples").glob("step-*/*.wav"))
    print("Status:", final_status.get("status"), "step:", final_status.get("step"))
    print("Validation:", final_status.get("validation"))
    print("Samples generated:", len(wavs), "| adapter_final exists:", (RUN / "adapter_final").is_dir())
    print("Remote backup:", backup_info)
    import math
    val_loss = final_status.get("validation", {}).get("full_validation_loss")
    valid_loss = isinstance(val_loss, (int, float)) and math.isfinite(val_loss)
    checkpoints = list(RUN.glob("checkpoint-*/COMPLETE.json"))
    if (final_status.get("status") == "smoke_complete" and (RUN / "adapter_final/adapter_model.safetensors").is_file()
            and checkpoints and valid_loss and backup_info.get("remote_snapshot_current")):
        from workflow_gate import publish_smoke_receipt
        publish_smoke_receipt(cfg, PROJECT_COMMIT, RUN)
        print("TEST PASSED: training, sealed checkpoint, finite validation loss, Hub backup and final adapter worked.")
        print("Synthesis samples:", len(wavs), "(inspect audio; synthesis quality is not established by this test).")
        print("Set TEST=False (EPOCH=2) and run again for the full second epoch.")
    else:
        raise RuntimeError("TEST INCOMPLETE: check status, validation and remote backup above before a full run")

Cell 5 — status, converted reference audio and generated samples

Reference playback also works after an audit-only run. Compare the Urdu transcript against the audio; the training smoke test cannot certify transcript accuracy or speech quality.

In [ ]:
from IPython.display import Audio, display

def optional_json(path):
    try:
        return json.loads(path.read_text()) if path.exists() else {}
    except (OSError, ValueError) as exc:
        print("Optional report unavailable:", path.name, type(exc).__name__)
        return {}

status_value = optional_json(RUN / "status.json")
print(status_value or ("Dataset audit only" if AUDIT_ONLY else "No training status recorded"))
for path in sorted((RUN / "samples").glob("step-*/*.wav"))[-3:]:
    print(path)
    try:
        display(Audio(filename=str(path)))
    except (OSError, ValueError) as exc:
        print("Optional playback unavailable:", type(exc).__name__)
backup = optional_json(RUN / "backup_status.json")
if backup:
    print("Remote backup:", backup)

# Play the exact converted references used by training (no extra dataset download).
source_report = RUN / "dataset_source.json"
if source_report.exists():
    source = optional_json(source_report)
    from hashlib import sha256
    recipe_bytes = json.dumps(source["recipe"], ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    data_root = Path(cfg["work_dir"]) / "datasets" / sha256(recipe_bytes).hexdigest()
    import csv
    with (data_root / "validation.csv").open(encoding="utf-8") as handle:
        references = list(csv.DictReader(handle))[:3]
    for row in references:
        print("Reference:", row["text"])
        display(Audio(filename=str(data_root / row["audio"])))

Cell 6 — optional synthesis from the final mixed adapter.
For another mixed epoch use a new run ID and INIT_FROM_RUN pointing to a completed mixed run. The notebook keeps its original source weights out of future optimizer resumes.


In [ ]:
GENERATE_FINAL_SAMPLE = False
if GENERATE_FINAL_SAMPLE and not AUDIT_ONLY and (RUN / "adapter_final").is_dir():
    output = RUN / "manual_sample.wav"
    subprocess.run([TRAIN_PYTHON, str(REPO / "synthesize.py"), "--adapter", str(RUN / "adapter_final"),
                    "--text", "آج موسم بہت خوشگوار ہے۔ ہم اردو بولنا سیکھ رہے ہیں۔", "--output", str(output)],
                   cwd=REPO, check=True)
    display(Audio(filename=str(output)))